In [13]:
%matplotlib

import torch
import numpy as np
import matplotlib.pyplot as plt

Using matplotlib backend: TkAgg


In [14]:
trunk_x_0 = torch.tensor([[0, 0, 0.3], [0, 0, 0.3]])
trunk_xd_0 = torch.zeros_like(trunk_x_0)
trunk_o_0 = torch.zeros_like(trunk_x_0)
trunk_od_0 = torch.zeros_like(trunk_x_0)

In [16]:
trunk_x_lo = torch.tensor([[0.02823, 0.07039, 0.1932], [0.12823, -0.12039, 0.2332]])
trunk_xd_lo = torch.tensor([[0.67559, 1.6845, 0.55862], [0.67559, 1.6845, 0.8862]])
trunk_o_lo = torch.tensor([[np.pi / 6, 0, 0], [np.pi / 4, 0, 0]])
trunk_od_lo = torch.tensor([[10, 0, 0], [10, 0, 0]])
t_th = torch.tensor([[0.45], [0.65]])

In [17]:
def BezierTrajectory(start: torch.Tensor, start_v: torch.Tensor, end: torch.Tensor, end_v: torch.Tensor, t_th: torch.Tensor, t_ex: float):

    w = torch.stack((
        start.unsqueeze(1),
        (start - (1. / 3.) * t_th * start_v).unsqueeze(1),
        (end - (1. / 3.) * t_th * end_v).unsqueeze(1),
        end.unsqueeze(1)), dim=2).squeeze()

    w_d = torch.stack((
        ((3 / t_th) * (w[:, 1] - w[:, 0])).unsqueeze(1),
        ((3 / t_th) * (w[:, 2] - w[:, 1])).unsqueeze(1),
        ((3 / t_th) * (w[:, 3] - w[:, 2])).unsqueeze(1)), dim=2).squeeze()

    t = t_ex / t_th

    bezier_curve_3 = torch.cat((
        (1.0) * (t**0) * (1 - t)**(3 - 0),
        (3.0) * (t**1) * (1 - t)**(3 - 1),
        (3.0) * (t**2) * (1 - t)**(3 - 2),
        (1.0) * (t**3) * (1 - t)**(3 - 3),
    ), dim=-1)

    bezier_curve_2 = torch.cat((
        (1.0) * (t**0) * (1 - t)**(2 - 0),
        (2.0) * (t**1) * (1 - t)**(2 - 1),
        (1.0) * (t**2) * (1 - t)**(2 - 2),
    ), dim=-1)

    bezier_point = torch.cat((
        (w[..., 0] * bezier_curve_3).sum(dim=-1).reshape(-1, 1),
        (w[..., 1] * bezier_curve_3).sum(dim=-1).reshape(-1, 1),
        (w[..., 2] * bezier_curve_3).sum(dim=-1).reshape(-1, 1)), dim=1)

    bezier_velocity = torch.cat((
        (w_d[..., 0] * bezier_curve_2).sum(dim=-1).reshape(-1, 1),
        (w_d[..., 1] * bezier_curve_2).sum(dim=-1).reshape(-1, 1),
        (w_d[..., 2] * bezier_curve_2).sum(dim=-1).reshape(-1, 1)), dim=1)

    return bezier_point, bezier_velocity

In [18]:
max_time = 0.655

In [19]:
pos_lin_vel = [BezierTrajectory(trunk_x_0, trunk_xd_0, trunk_x_lo, trunk_xd_lo, t_th, t) for t in torch.arange(0, max_time, 0.005)]
pos = torch.stack([val[0] for val in pos_lin_vel], dim=1)
lin_vel = torch.stack([val[1] for val in pos_lin_vel], dim=1)

orient_ang_vel = [BezierTrajectory(trunk_o_0, trunk_od_0, trunk_o_lo, trunk_od_lo, t_th, t) for t in torch.arange(0, max_time, 0.005)]
orient = torch.stack([val[0] for val in orient_ang_vel], dim=1)
ang_vel = torch.stack([val[1] for val in orient_ang_vel], dim=1)

In [20]:
pos.shape, lin_vel.shape

(torch.Size([2, 131, 3]), torch.Size([2, 131, 3]))

In [21]:
orient.shape, ang_vel.shape

(torch.Size([2, 131, 3]), torch.Size([2, 131, 3]))

In [22]:
# fig = plt.figure(figsize=(10, 10))
# ax = plt.axes(projection='3d')
# ax.set_zlim(0, 0.35)
# ax.set_xlim(-0.35, 0.35)
# ax.set_ylim(-0.35, 0.35)
# ax.scatter(pos[..., 0], pos[..., 1], pos[..., 2], color='blue')
# ax.scatter(trunk_x_0[:, 0], trunk_x_0[:, 1], trunk_x_0[:, 2], alpha=1, linewidths=5)
# ax.scatter(trunk_x_lo[:, 0], trunk_x_lo[:, 1], trunk_x_lo[:, 2], alpha=1, linewidths=5)
# ax.quiver(pos[..., 0], pos[..., 1], pos[..., 2], lin_vel[..., 0], lin_vel[..., 1], lin_vel[..., 2], length=0.1, normalize=False, color='red', arrow_length_ratio=0)
# plt.show()

In [23]:
plt.close()

In [24]:
fig = plt.figure(figsize=(10, 10))
ax = plt.axes(projection='3d')
ax.set_zlim(0, 0.4)
ax.set_xlim(-0.4, 0.4)
ax.set_ylim(-0.4, 0.4) 

i = 1
step_size = 5
draw_l_vel = False
ax.scatter(pos[i][..., 0], pos[i][..., 1], pos[i][..., 2], color='black', linewidths=1)
if draw_l_vel:
    ax.quiver(pos[i][..., 0], pos[i][..., 1], pos[i][..., 2],
              lin_vel[i][..., 0], lin_vel[i][..., 1], lin_vel[i][..., 2],
              length=0.1, color='purple', arrow_length_ratio=0)
ax.scatter(trunk_x_0[i, 0], trunk_x_0[i, 1], trunk_x_0[i, 2], alpha=1, linewidths=5)
ax.scatter(trunk_x_lo[i, 0], trunk_x_lo[i, 1], trunk_x_lo[i, 2], alpha=1, linewidths=5)

for j in range(0, len(orient[i]), step_size):
    # Getting the current position and orientation
    p = pos[i, j].numpy()
    o = orient[i, j].numpy()

    # Define the Cartesian frame vectors (x, y, z)
    x_vec = np.array([1, 0, 0])  # X-axis
    y_vec = np.array([0, 1, 0])  # Y-axis
    z_vec = np.array([0, 0, 1])  # Z-axis

    # Rotate the frame based on orientation
    rotation_matrix = np.eye(3)
    rotation_matrix = np.matmul(rotation_matrix, np.array([
        [1, 0, 0],
        [0, np.cos(o[0]), -np.sin(o[0])],
        [0, np.sin(o[0]), np.cos(o[0])]
    ]))
    rotation_matrix = np.matmul(rotation_matrix, np.array([
        [np.cos(o[1]), 0, np.sin(o[1])],
        [0, 1, 0],
        [-np.sin(o[1]), 0, np.cos(o[1])]
    ]))
    rotation_matrix = np.matmul(rotation_matrix, np.array([
        [np.cos(o[2]), -np.sin(o[2]), 0],
        [np.sin(o[2]), np.cos(o[2]), 0],
        [0, 0, 1]
    ]))

    x_vec = np.matmul(rotation_matrix, x_vec)
    y_vec = np.matmul(rotation_matrix, y_vec)
    z_vec = np.matmul(rotation_matrix, z_vec)

    # Plotting the frame
    ax.quiver(p[0], p[1], p[2], x_vec[0], x_vec[1], x_vec[2], length=0.05, color='r', arrow_length_ratio=0)
    ax.quiver(p[0], p[1], p[2], y_vec[0], y_vec[1], y_vec[2], length=0.05, color='g', arrow_length_ratio=0)
    ax.quiver(p[0], p[1], p[2], z_vec[0], z_vec[1], z_vec[2], length=0.05, color='b', arrow_length_ratio=0)

plt.show()

In [25]:
0.5 > t_th

tensor([[ True],
        [False]])

In [30]:
torch.where(0.5 > t_th)[0]

tensor([0])